# Anchor-projection features from review embeddings

Raw 384-dim embeddings (and their PCA compression) *worsened* both price and
rating models — too wide/noisy for XGBoost on ~113k rows. This notebook takes
the **interpretable** route instead: project each review embedding onto a small
set of named wine concepts and keep only those similarity scores as features.

Method (**multi-anchor**):
1. Each concept in `ANCHORS_MULTI` is described by several anchor sentences.
2. Encode the anchors with the *same* model as `06` (`all-MiniLM-L6-v2`),
   average them into one L2-normalised **concept centroid** (averaging several
   phrasings is more robust than a single anchor sentence).
3. Cosine-similarity of every (cached) review embedding against each centroid
   → one `anchor_<concept>` feature per concept.

Output: `features_embeddings_anchored.parquet`, keyed by `wine_id`, joinable to
`features_basic` like the other feature modules. These are the semantic analog
of the lexical `features_keywords` — same idea, but meaning-based not word-match.

In [ ]:
import numpy as np
import pandas as pd
import itables
from itables import show
from sentence_transformers import SentenceTransformer

itables.options.columnDefs = [{"className": "dt-left", "targets": "_all"}]

EMBEDDINGS_PATH = r"..\..\.data\features_embeddings.parquet"   # from 06_nlp_embeddings
ANCHORED_PATH   = r"..\..\.data\features_embeddings_anchored.parquet"
SILVER_PATH     = r"..\..\.data\wine_reviews_silver.parquet"   # for the sanity check only
MODEL_NAME      = "all-MiniLM-L6-v2"

emb = pd.read_parquet(EMBEDDINGS_PATH)
emb_cols = [c for c in emb.columns if c.startswith("emb_")]
print(f"embeddings: {emb.shape}  ({len(emb_cols)} dims)")
assert emb["wine_id"].is_unique, "wine_id must be unique"

## Anchor concepts

Each concept is defined by a few example sentences. Edit freely — everything
below is driven by this dict.

In [ ]:
ANCHORS_MULTI = {
    "dry": [
        "This wine is bone dry.",
        "A dry style with no residual sugar.",
        "Crisp and dry on the finish.",
    ],
    "acidic": [
        "Bright, zesty acidity.",
        "Crisp acidity drives the palate.",
        "Vibrant, mouthwatering acidity.",
    ],
    "sweet": [
        "This wine is sweet and lush.",
        "Noticeable residual sugar on the palate.",
        "A rich, sugary sweetness.",
    ],
    "alcohol": [
        "A hot, boozy finish.",
        "Noticeable alcoholic warmth.",
        "High alcohol gives a burning sensation.",
    ],

    "citrus_fruits": [
        "Aromas of lemon zest.",
        "Notes of orange peel.",
        "Bright grapefruit and citrus flavors.",
    ],
    "green_fruits": [
        "Notes of green apple.",
        "Crisp pear flavors.",
        "Orchard fruit aromas of apple and pear.",
    ],
    "red_fruits": [
        "Flavors of ripe strawberry.",
        "Bright cherry notes.",
        "Juicy raspberry fruit.",
    ],
    "black_fruits": [
        "Rich blackberry aromas.",
        "Dark notes of black cherry.",
        "Concentrated black currant fruit.",
    ],
    "tropical_fruits": [
        "Exotic aromas of pineapple.",
        "Ripe melon notes.",
        "Notes of banana and guava.",
    ],
    "stone_fruits": [
        "Juicy peach flavors.",
        "Aromas of ripe apricot.",
        "Notes of nectarine.",
    ],

    "oak": [
        "Notes of oak on the finish.",
        "Toasted wood aromas.",
        "Aged in oak barrels, with woody notes.",
    ],
    "coconut": [
        "Aromas of coconut.",
        "Notes of toasted coconut.",
        "A coconut-like creaminess from oak aging.",
    ],
    "vanilla": [
        "Sweet vanilla notes.",
        "Aromas of vanilla bean.",
        "Vanilla and baking spice from oak.",
    ],
    "smoke": [
        "Smoky aromas.",
        "Notes of charred wood.",
        "A campfire-like smokiness.",
    ],

    "tannic": [
        "Firm, grippy tannins.",
        "Structured, drying tannins.",
        "Robust tannic backbone.",
    ],
    "body": [
        "A full-bodied wine.",
        "Rich, weighty texture.",
        "Light-bodied and delicate on the palate.",
    ],

    "flowers": [
        "Floral aromas of violet.",
        "Notes of rose petal.",
        "Delicate jasmine and blossom aromas.",
    ],
    "herbs": [
        "Herbal notes of thyme.",
        "Aromas of fresh sage.",
        "Notes of fresh-cut grass.",
    ],
    "spicy": [
        "Peppery spice notes.",
        "Aromas of clove.",
        "Notes of black pepper.",
    ],
    "vegetables": [
        "Green, vegetal notes.",
        "Aromas of bell pepper.",
        "Notes of tomato leaf.",
    ],
    "earth": [
        "Earthy mineral notes.",
        "Aromas of wet stone.",
        "Notes of forest floor.",
    ],
    "leather": [
        "Notes of leather.",
        "Aromas of game and leather.",
        "A leathery, savory complexity.",
    ],
}

concepts = list(ANCHORS_MULTI)
print(f"{len(concepts)} concepts, "
      f"{sum(len(v) for v in ANCHORS_MULTI.values())} anchor sentences")

## Build concept centroids

Encode every anchor sentence with the same model as `06`, then per concept
average the (unit) anchor vectors and re-normalise → one direction per concept.

In [ ]:
model = SentenceTransformer(MODEL_NAME)
print(f"model dim: {model.get_sentence_embedding_dimension()}")

centroids = []
for c in concepts:
    a = model.encode(ANCHORS_MULTI[c], convert_to_numpy=True, normalize_embeddings=True)
    v = a.mean(axis=0)
    v = v / (np.linalg.norm(v) + 1e-12)
    centroids.append(v)

A = np.vstack(centroids).astype(np.float32)         # (n_concepts, dim)
assert A.shape[1] == len(emb_cols), "anchor dim != embedding dim"
print(f"centroid matrix A: {A.shape}")

## Project reviews onto the anchors

L2-normalise the cached review embeddings, then a single matrix multiply gives
the cosine similarity of each review to each concept centroid.

In [ ]:
E = emb[emb_cols].to_numpy(np.float32)
E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)

sims = (E @ A.T).astype(np.float32)                 # (N, n_concepts) cosine in [-1, 1]
anchor_cols = [f"anchor_{c}" for c in concepts]
anchor_df = pd.DataFrame(sims, columns=anchor_cols, index=emb.index)
print(f"anchor features: {anchor_df.shape}")
anchor_df.describe().T[["mean", "std", "min", "max"]].round(3)

## Sanity check — top concepts per review

For a few sample wines, the highest-scoring concepts should match what the
review actually says. Pulls the review text from Silver (join on `wine_id`).

In [ ]:
silver = pd.read_parquet(SILVER_PATH)[["wine_id", "review"]]
sample = anchor_df.join(emb["wine_id"]).sample(6, random_state=7)
sample = sample.merge(silver, on="wine_id", how="left")

for _, row in sample.iterrows():
    top = row[anchor_cols].astype(float).sort_values(ascending=False).head(4)
    tags = ", ".join(f"{c.replace('anchor_', '')} {v:.2f}" for c, v in top.items())
    print(f"- {str(row['review'])[:170]}")
    print(f"    -> {tags}\n")

## Save

`wine_id` + 22 `anchor_*` cosine features. Joins to `features_basic` on
`wine_id`; use in `models/01_models_retail.ipynb` / `02_models_rating.ipynb`
as a `base + anchors` model alongside `base + kw`.

In [ ]:
out = pd.concat([emb[["wine_id"]], anchor_df], axis=1)
out.to_parquet(ANCHORED_PATH, index=False)
size_mb = __import__("os").path.getsize(ANCHORED_PATH) / 1e6
print(f"Saved {out.shape[0]:,} rows x {out.shape[1]} cols ({size_mb:.1f} MB) -> {ANCHORED_PATH}")
out.head()